<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/speech_to_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import re
import librosa
import torch
import pandas as pd
import torchaudio
import numpy as np
from torch.utils.data import DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MAX_AUDIO_LENGTH = 160000  # Maximum audio length in samples (can be adjusted)

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Directory Paths
MAIN_DIR = '/content/drive/MyDrive/KartalOl Corpus/Dataset/Speech Recognition dataset/Work with Farhan/dataset'




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
os.listdir(MAIN_DIR)

['sentences', 'voices']

In [9]:
import os
import re
import pandas as pd
import librosa
import torch
import torchaudio
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

class AZBDataset(Dataset):
    """Custom Dataset for South Azerbaijani Speech-to-Text task."""
    def __init__(self, main_dir, sampling_rate=16000):
        self.dataset = []
        self.sampling_rate = sampling_rate
        self.person_folders = self.get_person_folders(main_dir)
        self.name_csv_dir = os.path.join(main_dir, 'sentences')
        self.csv_dict = self.load_csv_files(self.name_csv_dir)
        self.prepare_dataset()

    def get_person_folders(self, main_dir):
        """Retrieve all person folders ending with '-bot'."""
        return [
            os.path.join(main_dir, folder)
            for folder in os.listdir(main_dir+'/voices')
            if os.path.isdir(os.path.join(main_dir, folder)) and folder.endswith('-bot')
        ]

    def load_csv_files(self, name_csv_dir):
        """Load CSV files into a dictionary with PersonID as keys."""
        csv_dict = {}
        for file in os.listdir(name_csv_dir):
            if file.endswith('.csv'):
                person_id = os.path.splitext(file)[0]
                csv_path = os.path.join(name_csv_dir, file)
                try:
                    df = pd.read_csv(csv_path)
                    csv_dict[person_id] = df
                except Exception as e:
                    print(f"Error reading {csv_path}: {e}")
        return csv_dict

    def normalize_text(self, text):
        """Normalize text by lowercasing and removing punctuation."""
        text = text.lower()
        text = text.translate(str.maketrans('', '', '""'))
        text = ' '.join(text.split())  # Remove extra whitespace
        return text

    def prepare_dataset(self):
        for person_folder in self.person_folders:
            person_id = os.path.basename(person_folder).replace('-bot', '')
            if person_id not in self.csv_dict:
                print(f"CSV for PersonID {person_id} not found. Skipping this person.")
                continue
            df = self.csv_dict[person_id]
            for file in os.listdir(person_folder):
                if file.endswith('.wav'):
                    try:
                        file_parts = os.path.splitext(file)[0].split('_')
                        if len(file_parts) != 4:
                            print(f"Filename {file} does not match the expected format. Skipping.")
                            continue

                        file_person_id, sentence_id, number, book_id = file_parts
                        if 'ID' in sentence_id:
                            sentence_id = int(re.search(r'ID(\d+)', sentence_id).group(1))

                        if file_person_id != person_id:
                            print(f"PersonID mismatch in file {file}. Skipping.")
                            continue

                        matching_rows = df.iloc[sentence_id]
                        text = matching_rows['Sentence']
                        normalized_text = self.normalize_text(text)
                        audio_path = os.path.join(person_folder, file)

                        waveform, sr = librosa.load(audio_path, sr=None)
                        if sr != self.sampling_rate:
                            print(f"Sample rate for {file} is {sr}, resampling to {self.sampling_rate}")
                            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=self.sampling_rate)

                        self.dataset.append({
                            'audio_path': audio_path,
                            'text': normalized_text
                        })
                    except Exception as e:
                        print(f"Error processing file {file}: {e}")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data = self.dataset[idx]
        audio_path = data['audio_path']
        text = data['text']
        waveform, sr = librosa.load(audio_path, sr=self.sampling_rate)
        return waveform, text

class WhisperSTTModel:
    def __init__(self, model_name="openai/whisper-large-v2"):
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

    def train(self, dataset, epochs=10, batch_size=8, learning_rate=1e-5, save_path="stt_model.pt"):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(device)
        self.model.train()

        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=learning_rate)

        for epoch in range(epochs):
            total_loss = 0
            for waveforms, texts in dataloader:
                waveforms = [torch.tensor(wf).float().unsqueeze(0) for wf in waveforms]
                inputs = self.processor(waveforms, return_tensors="pt", sampling_rate=16000, padding=True)
                labels = self.processor(texts, return_tensors="pt", padding=True).input_ids

                inputs = {key: val.to(device) for key, val in inputs.items()}
                labels = labels.to(device)

                outputs = self.model(**inputs, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                total_loss += loss.item()
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}")
            torch.save(self.model.state_dict(), save_path)

    def predict(self, audio_path):
        waveform, sr = librosa.load(audio_path, sr=16000)
        inputs = self.processor(torch.tensor(waveform).float(), return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            predicted_ids = self.model.generate(inputs.input_features)
        transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        return transcription

    def evaluate(self, dataset):
        self.model.eval()
        total_loss = 0
        dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

        with torch.no_grad():
            for waveforms, texts in dataloader:
                waveforms = [torch.tensor(wf).float().unsqueeze(0) for wf in waveforms]
                inputs = self.processor(waveforms, return_tensors="pt", sampling_rate=16000, padding=True)
                labels = self.processor(texts, return_tensors="pt", padding=True).input_ids

                inputs = {key: val.to('cuda') for key, val in inputs.items()}
                labels = labels.to('cuda')

                outputs = self.model(**inputs, labels=labels)
                loss = outputs.loss
                total_loss += loss.item()
        print(f"Evaluation Loss: {total_loss/len(dataloader)}")




In [10]:

dataset = AZBDataset(MAIN_DIR)
model = WhisperSTTModel()
model.train(dataset, epochs=6, batch_size=4, save_path="stt_model.pt")



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/KartalOl Corpus/Dataset/Speech Recognition dataset/Work with Farhan/dataset/Name_csv'

In [8]:
# Test on a single file
test_path = "/path/to/your/test/audio.wav"
transcription = model.predict(test_path)
print("Transcription:", transcription)

Creating dataset...


ValueError: axes don't match array

In [ ]:
# Example Transcription
sample_audio_path = '/path/to/your/audio/file.wav'
transcription = transcribe_audio(sample_audio_path)
print("Transcription:", transcription)